# Working with results

Everything you can get out of a cut, a sampler run, the scikit-learn estimator and link
communities: the cluster of each point, the members and score of each cluster, where each
cluster sits in the tree, and for networks the communities of each node.

This notebook needs `pip install adaptive_cut scikit-learn networkx`.

In [1]:
import networkx as nx
import numpy as np
from scipy.cluster.hierarchy import fcluster, linkage, to_tree

import adaptive_cut as ac

In [2]:
def blobs(seed=0):
    """Three tight blobs (150 points each) next to two wide ones (300 each)."""
    rng = np.random.default_rng(seed)
    centers = np.array([[0, 0], [1.2, 0], [0.6, 1.0], [8, 0], [4, 7]])
    std = np.array([0.25, 0.25, 0.25, 1.6, 1.6])
    y = np.repeat(np.arange(5), [150, 150, 150, 300, 300])
    X = centers[y] + std[y, None] * rng.standard_normal((len(y), 2))
    return X, y


X, y = blobs(seed=0)
Z = linkage(X, "ward")
scores = ac.gaussian_scores(Z, X)
level = ac.best_level_cut(Z, scores)
res = ac.adaptive_cut(Z, scores, T=160, n_steps=100_000, rng=0)

## The sampler run: `AdaptiveCutResult`

`adaptive_cut` returns the best cut visited, the last state, the starting cut and a trace of
the objective. `res.labels`, `res.value`, `res.n_clusters` and `res.nodes` are those of
`res.best`.

In [3]:
print(res)
print("best:", res.best, "| last:", res.final, "| start:", res.initial)
print(
    f"gain over the start {res.gain:.1f}, acceptance rate {res.acceptance_rate:.3f}, "
    f"{res.n_accepted} accepted moves of {res.n_steps}"
)
print(f"temperature {res.T}, annealing from step {res.k_sa}, engine {res.engine!r}")
print("trace: objective every", res.trace_every, "steps,", len(res.trace), "values")

AdaptiveCutResult(value=3225.84, n_clusters=5, initial=3103.51, n_steps=100000, T=160, engine='numba')
best: Cut(n_clusters=5, value=3225.84) | last: Cut(n_clusters=5, value=3225.84) | start: Cut(n_clusters=3, value=3103.51, height=31.6511)
gain over the start 122.3, acceptance rate 0.363, 36289 accepted moves of 100000
temperature 160.0, annealing from step 66666, engine 'numba'
trace: objective every 1 steps, 100000 values


## A cut: `Cut`

- `labels[i]`: cluster of point `i`. Labels are canonical (point 0 is in cluster 0, clusters
  numbered in order of first appearance), so two cuts give the same partition exactly when
  their labels are equal.
- `nodes[c]`: the tree node of cluster `c` (scipy numbering: leaves `0..n-1`, row `i` of `Z`
  creates node `n + i`).
- `value`: the objective, the sum of the scores of the cut's nodes.
- `sizes`, `n_clusters`; `height` is set only for single-level cuts.

In [4]:
cut = res.best
print("labels of the first 10 points:", cut.labels[:10])
print("tree nodes:", cut.nodes, "| sizes:", cut.sizes, "| value:", round(cut.value, 2))
print("value == sum of node scores:", np.isclose(cut.value, scores[cut.nodes].sum()))
print("single-level cut height:", level.height, "| adaptive cut height:", cut.height)

labels of the first 10 points: [0 0 0 0 0 0 0 0 0 0]
tree nodes: [2055 2047 2051 2095 2096] | sizes: [154 144 152 296 304] | value: 3225.84
value == sum of node scores: True
single-level cut height: 31.651112055631426 | adaptive cut height: None


## Members, score and height of each cluster

In [5]:
n = len(X)
parent_height = np.zeros(2 * n - 1)  # height at which each node is merged away
parent_height[Z[:, :2].astype(int).ravel()] = np.repeat(Z[:, 2], 2)
for c, v in enumerate(cut.nodes):
    members = np.flatnonzero(cut.labels == c)  # points of cluster c
    born = Z[v - n, 2] if v >= n else 0.0  # height where the cluster is complete
    print(
        f"cluster {c}: node {v}, {len(members)} points (first {members[:4].tolist()}), "
        f"score {scores[v]:8.1f}, exists between heights {born:.2f} and {parent_height[v]:.2f}"
    )

cluster 0: node 2055, 154 points (first [0, 1, 2, 3]), score    720.6, exists between heights 3.67 and 14.95
cluster 1: node 2047, 144 points (first [23, 300, 301, 302]), score    688.9, exists between heights 3.12 and 14.11
cluster 2: node 2051, 152 points (first [135, 150, 151, 152]), score    721.0, exists between heights 3.29 and 14.11
cluster 3: node 2095, 296 points (first [450, 451, 452, 453]), score    550.1, exists between heights 30.04 and 138.33
cluster 4: node 2096, 304 points (first [497, 519, 525, 623]), score    545.2, exists between heights 31.65 and 138.33


## Points under any tree node

`scipy.cluster.hierarchy.to_tree` gives every node of the tree; `pre_order()` lists its leaves.

In [6]:
root, tree_nodes = to_tree(Z, rd=True)
v = cut.nodes[0]
print(
    f"node {v} has {tree_nodes[v].get_count()} points:",
    sorted(tree_nodes[v].pre_order())[:10],
    "...",
)

node 2055 has 154 points: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] ...


## Comparing two cuts

In [7]:
table = np.zeros((level.n_clusters, cut.n_clusters), dtype=int)
np.add.at(table, (level.labels, cut.labels), 1)
print("rows: single-level clusters, columns: adaptive clusters")
print(table)
print(
    "same partition as fcluster at the single-level height:",
    len(set(zip(fcluster(Z, level.height, "distance"), level.labels))) == level.n_clusters,
)

rows: single-level clusters, columns: adaptive clusters
[[154 144 152   0   0]
 [  0   0   0 296   0]
 [  0   0   0   0 304]]
same partition as fcluster at the single-level height: True


## The scikit-learn estimator

`AdaptiveClustering` stores the tree, the scores, the cut and the sampler run.

In [8]:
model = ac.AdaptiveClustering(linkage="ward", T=160, n_steps=100_000, random_state=0).fit(X)
print("labels_:", model.labels_[:10], "| n_clusters_:", model.n_clusters_)
print("cut_:", model.cut_, "| result_:", model.result_)
print("linkage_matrix_:", model.linkage_matrix_.shape, "| scores_:", model.scores_.shape)
print("children_:", model.children_[:2].tolist(), "| distances_:", model.distances_[:2])
print("same as the step-by-step run:", np.array_equal(model.labels_, res.labels))

labels_: [0 0 0 0 0 0 0 0 0 0] | n_clusters_: 5
cut_: Cut(n_clusters=5, value=3225.84) | result_: AdaptiveCutResult(value=3225.84, n_clusters=5, initial=3103.51, n_steps=100000, T=160, engine='numba')
linkage_matrix_: (1049, 4) | scores_: (2099,)
children_: [[93, 104], [60, 127]] | distances_: [0.00300367 0.00479947]
same as the step-by-step run: True


## Link communities: `LinkCommunities`

Leaf `i` of the edge tree is edge `r.edges[i]` (rows of node indices into `r.node_labels`).

In [9]:
G = nx.karate_club_graph()
r = ac.link_communities(G, rng=0)
print(r)
print("nodes:", r.node_labels[:5], "| edges:", r.edges[:3].tolist(), "| weights:", r.weights)
print("cuts:", r.level_cut, r.cut, "| sampler:", r.adaptive)
print(f"partition density {r.partition_density:.3f}, balancedness {r.balancedness:.2f}, T {r.T:g}")

LinkCommunities(N=34, M=78, balancedness=0.631, D_level=0.2848, D=0.4030, n_communities=29)
nodes: [0 1 2 3 4] | edges: [[0, 1], [0, 2], [0, 3]] | weights: None
cuts: Cut(n_clusters=22, value=0.284758, height=0.642857) Cut(n_clusters=29, value=0.402991) | sampler: AdaptiveCutResult(value=0.402991, n_clusters=29, initial=0.284758, n_steps=10000, T=0.001, engine='numba')
partition density 0.403, balancedness 0.63, T 0.001


### Community of each edge

In [10]:
for e in range(5):
    u, v = r.node_labels[r.edges[e]]
    print(f"edge {u}-{v}: community {r.cut.labels[e]}")

edge 0-1: community 0
edge 0-2: community 0
edge 0-3: community 0
edge 0-4: community 1
edge 0-5: community 1


### Members of each community, and its partition density contribution

In [11]:
for c, members in enumerate(r.communities()):
    if len(members) >= 4:
        n_edges = int(r.cut.sizes[c])
        print(
            f"community {c}: {len(members)} nodes {members.tolist()}, {n_edges} edges, "
            f"contribution to D {r.scores[r.cut.nodes[c]]:.3f}"
        )

community 0: 6 nodes [0, 1, 2, 3, 7, 13], 14 edges, contribution to D 0.162
community 1: 6 nodes [0, 4, 5, 6, 10, 16], 10 edges, contribution to D 0.064
community 5: 4 nodes [0, 1, 17, 21], 4 edges, contribution to D 0.017
community 14: 4 nodes [8, 30, 32, 33], 6 edges, contribution to D 0.077
community 17: 7 nodes [14, 15, 18, 20, 22, 32, 33], 10 edges, contribution to D 0.034
community 21: 5 nodes [23, 26, 29, 32, 33], 5 edges, contribution to D 0.011


### Communities of each node, with the share of its edges

In [12]:
M = r.memberships()  # sparse (nodes x communities), rows sum to 1
node_communities = {
    node: {int(c): round(float(M[[i]].toarray()[0, c]), 2) for c in M[[i]].nonzero()[1]}
    for i, node in enumerate(r.node_labels)
}
for node in [0, 33, 8]:
    print(f"node {node}:", node_communities[node])

node 0: {0: 0.31, 1: 0.25, 2: 0.06, 3: 0.06, 4: 0.06, 5: 0.12, 6: 0.06, 7: 0.06}
node 33: {14: 0.18, 15: 0.12, 16: 0.12, 17: 0.29, 21: 0.18, 25: 0.06, 28: 0.06}
node 8: {2: 0.2, 10: 0.2, 14: 0.6}


### Back into networkx

In [13]:
nx.set_edge_attributes(
    G, {tuple(r.node_labels[e]): int(c) for e, c in zip(r.edges, r.cut.labels)}, "community"
)
print(list(G.edges(data="community"))[:5])

[(0, 1, 0), (0, 2, 0), (0, 3, 0), (0, 4, 1), (0, 5, 1)]


### Another cut of the same tree

`r.Z` and `r.scores` are the edge tree and its scores, so the adaptive cut can be run again with
other settings.

In [14]:
again = ac.adaptive_cut(r.Z, r.scores, T=1e-3, n_steps=50_000, rng=1)
print(again, "| same partition:", np.array_equal(again.labels, r.cut.labels))

AdaptiveCutResult(value=0.402991, n_clusters=28, initial=0.284758, n_steps=50000, T=0.001, engine='numba') | same partition: False
